# HouseCrafter: 2D Floorplan to 3D Indoor Scene (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sourman-dev/houseCrafter/blob/feat/gradio-colab-ui/notebooks/HouseCrafter_Gradio_Colab.ipynb)
[![Project Page](https://img.shields.io/badge/Project-Page-blue)](https://neu-vi.github.io/houseCrafter/)

**Do not** `pip install -r requirements.txt` on Colab. That file pins `torch==2.1.0`, `flash-attn`, and `pytorch3d` and will spend 10+ minutes compiling (or fail). This notebook keeps Colab's preinstalled CUDA PyTorch and only installs Gradio/3D wheels.

Runtime: **GPU** (T4 is enough for `--mock` UI). Push `feat/gradio-colab-ui` to GitHub before running Step 3.

## Step 1 — GPU check

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))

## Step 2 — Mount Google Drive
Creates `MyDrive/Gradio/houseCrafter/output` if missing. Safe to re-run if Drive is already mounted.

In [ ]:
import os
from pathlib import Path

GDRIVE_OUT = Path("/content/drive/MyDrive/Gradio/houseCrafter/output")

try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    else:
        print("Drive already mounted")
    GDRIVE_OUT.mkdir(parents=True, exist_ok=True)
    print("OK Drive output:", GDRIVE_OUT)
except Exception as exc:
    GDRIVE_OUT = Path("/content/houseCrafter_output")
    GDRIVE_OUT.mkdir(parents=True, exist_ok=True)
    print("Drive unavailable, local fallback:", GDRIVE_OUT, "|", exc)

## Step 3 — Fetch this repo
Clones `sourman-dev/houseCrafter` and checks out `feat/gradio-colab-ui`. Skips clone if `app.py` is already on disk.

In [ ]:
import os
from pathlib import Path

REPO = Path("/content/houseCrafter")
BRANCH = "feat/gradio-colab-ui"
URL = "https://github.com/sourman-dev/houseCrafter.git"

if (REPO / "app.py").exists():
    print("Repo already present:", REPO)
else:
    %cd /content
    !git clone --branch {BRANCH} --single-branch {URL} houseCrafter || git clone {URL} houseCrafter

%cd /content/houseCrafter
!git fetch origin {BRANCH} || true
!git checkout {BRANCH} || echo "WARN: branch {BRANCH} missing on remote; using $(git branch --show-current)"
!git log -1 --oneline
assert Path("/content/houseCrafter/app.py").exists(), "app.py missing — push feat/gradio-colab-ui first"

## Step 4 — Install Colab overlay (wheels only, ~1–3 min)

This uses `requirements-colab.txt` + `scripts/colab_setup.sh`.

**Never run these on Colab:**
- `pip install -r requirements.txt` (pins torch 2.1 / flash-attn / pytorch3d)
- `pip install git+https://github.com/facebookresearch/pytorch3d.git` (compiles 10+ min)
- `sed` stripping version pins then installing everything

In [ ]:
%cd /content/houseCrafter
!bash scripts/colab_setup.sh

import gradio
import trimesh
print("gradio", gradio.__version__)
print("trimesh", trimesh.__version__)
try:
    import open3d
    print("open3d", open3d.__version__)
except Exception as exc:
    print("open3d optional miss:", exc)

## Step 5 — Optional checkpoints
Skip this cell to launch the UI in `--mock` mode (no 24GB weights). Run it only when you want full diffusion.

In [ ]:
%cd /content/houseCrafter
import os
from pathlib import Path

ckpt_cache = Path("/content/drive/MyDrive/houseCrafter_ckpts")
Path("ckpts").mkdir(exist_ok=True)
Path("dataRelease").mkdir(exist_ok=True)

if ckpt_cache.exists():
    print("Copying checkpoints from Drive cache...")
    !cp -r /content/drive/MyDrive/houseCrafter_ckpts/. ckpts/
else:
    print("No Drive cache. Skipping weight download for mock UI.")
    print("To fetch official weights later:")
    print("  !gdown --folder https://drive.google.com/drive/folders/1OY_V9nV5kOfGLa6oSlZMzVp0vRst2g3Y -O ckpts/")

print("ckpts:", os.listdir("ckpts")[:8] if Path("ckpts").exists() else [])

## Step 6 — Launch Gradio (`--mock`)
Opens a public `*.gradio.live` URL. Outputs sync to Drive `Gradio/houseCrafter/output`.

After checkpoints exist, relaunch without `--mock`.

In [ ]:
%cd /content/houseCrafter
import os
from pathlib import Path

out = "/content/drive/MyDrive/Gradio/houseCrafter/output"
if not Path(out).exists():
    out = "/content/houseCrafter_output"
    Path(out).mkdir(parents=True, exist_ok=True)

os.environ["GDRIVE_OUTPUT_DIR"] = out
!python app.py --mock --share --server_name 0.0.0.0 --gdrive_dir "{out}"

## Step 7 — List synced Drive outputs

In [ ]:
from pathlib import Path

out = Path("/content/drive/MyDrive/Gradio/houseCrafter/output")
if not out.exists():
    out = Path("/content/houseCrafter_output")
print("listing", out)
if out.exists():
    items = sorted(p.name for p in out.iterdir())
    print("count", len(items))
    for name in items[-8:]:
        print(" ", name)
else:
    print("no outputs yet")